In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
import keras
keras.utils.set_random_seed(28)
import datetime
from icecream import ic
from itertools import product
from sklearn.preprocessing import MinMaxScaler

I0000 00:00:1776268373.210709   16412 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776268373.212163   16412 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776268373.298585   16412 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776268375.907623   16412 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

In [2]:
CUSTOM_DATASET_PATH = Path("custom_data/")

In [3]:
merged = pd.read_csv(CUSTOM_DATASET_PATH / "merged.csv")
merged

,business_id,latitude,longitude,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
2,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,3,8,2,2,5,0,18
3,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
4,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
511134,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
511135,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
511136,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [4]:
weekdays = ("Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday")
merged = merged.drop(columns=[weekday.lower() + suffix for weekday, suffix in product(weekdays, ("_opening", "_closing", "_opening_length_hours"))])


In [5]:
merged = merged.drop(columns=["latitude", "longitude"])

In [6]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(merged.isnull().sum())

business_id                                         0
stars_business                                      0
review_count_review                                 0
Fuzhou_category                                     0
Themed Cafes_category                               0
Burmese_category                                    0
Falafel_category                                    0
Cafes_category                                      0
Conveyor Belt Sushi_category                        0
Australian_category                                 0
Sushi Bars_category                                 0
Mongolian_category                                  0
New Mexican Cuisine_category                        0
Poutineries_category                                0
Ethiopian_category                                  0
Hong Kong Style Cafe_category                       0
Cafeteria_category                                  0
Chicken Shop_category                               0
Vegetarian_category         

In [7]:
merged.info(max_cols=500)

<class 'pandas.DataFrame'>
RangeIndex: 511138 entries, 0 to 511137
Data columns (total 388 columns):
 #    Column                                            Non-Null Count   Dtype  
---   ------                                            --------------   -----  
 0    business_id                                       511138 non-null  str    
 1    stars_business                                    511138 non-null  float64
 2    review_count_review                               511138 non-null  int64  
 3    Fuzhou_category                                   511138 non-null  int64  
 4    Themed Cafes_category                             511138 non-null  int64  
 5    Burmese_category                                  511138 non-null  int64  
 6    Falafel_category                                  511138 non-null  int64  
 7    Cafes_category                                    511138 non-null  int64  
 8    Conveyor Belt Sushi_category                      511138 non-null  int64  
 9    Au

In [8]:
timestamp_format = "%Y-%m-%d %H:%M:%S"
merged["date"] = merged["date"].apply(lambda x: datetime.datetime.strptime(x, timestamp_format))
merged["yelping_since"] = merged["yelping_since"].apply(lambda x: datetime.datetime.strptime(x, timestamp_format))
merged["date"] = (merged["date"] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
merged["yelping_since"] = (merged["yelping_since"] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')

In [9]:
merged_no_ids = merged.drop(labels=["review_id", "user_id", "business_id"], axis=1)
display(merged_no_ids)

,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,Conveyor Belt Sushi_category,Australian_category,Sushi Bars_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,4.0,80,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,4.0,80,0,0,0,0,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
2,4.0,80,0,0,0,0,0,0,0,0,...,0,0,0,3,8,2,2,5,0,18
3,4.0,80,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
4,4.0,80,0,0,0,0,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,4.5,35,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
511134,4.5,35,0,0,0,0,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
511135,4.5,35,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
511136,4.5,35,0,0,0,0,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [10]:
merged_only_ids = merged[["review_id", "user_id", "business_id"]]
merged_only_ids

,review_id,user_id,business_id
0,BXQcBN0iAi1lAUxibGLFzA,6_SpY41LIHZuIaiDs5FMKA,MTSW4McQd7CbVtyjqoe9mw
1,uduvUCvi9w3T2bSGivCfXg,tCXElwhzekJEH6QJe3xs7Q,MTSW4McQd7CbVtyjqoe9mw
2,a0vwPOqDXXZuJkbBW2356g,WqfKtI-aGMmvbA9pPUxNQQ,MTSW4McQd7CbVtyjqoe9mw
3,MKNp_CdR2k2202-c8GN5Dw,3-1va0IQfK-9tUMzfHWfTA,MTSW4McQd7CbVtyjqoe9mw
4,D1GisLDPe84Rrk_R4X2brQ,EouCKoDfzaVG0klEgdDvCQ,MTSW4McQd7CbVtyjqoe9mw
...,...,...,...
511133,qBcwQEQPnLxjkw-xbUIF4Q,6nF5PT1c0dF6EpOgQdF2tw,WnT9NIzQgLlILjPT0kEcsQ
511134,G8fbysnUAUmqq1XWTjMQ4Q,1M78_w4J9f5S8xmUVYyxdQ,WnT9NIzQgLlILjPT0kEcsQ
511135,JKiy0aeyGd3KmXN7uRPFLw,B7TD5yTemGv50y4wM2EVNA,WnT9NIzQgLlILjPT0kEcsQ
511136,JITY01bGbdsiUBznLz9rdg,HI8QwhpeP_ZRY5JZy11VDw,WnT9NIzQgLlILjPT0kEcsQ


In [13]:
user = pd.read_csv(CUSTOM_DATASET_PATH / "user_for_vis.csv")
user

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,fans,average_stars,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,267,3.91,...,55,56,18,232,844,467,467,239,180,14995
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...",3138,3.74,...,184,157,251,1847,7054,3131,3131,1521,1946,4646
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013",52,3.32,...,10,17,3,66,96,119,119,35,18,381
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011",28,4.27,...,1,6,2,12,16,26,26,10,9,131
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,NaN,1,3.54,...,0,0,0,1,1,0,0,0,0,27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1987892,fB3jbHi3m0L2KgGOxBv6uw,Jerrold,23,2015-01-06 00:31:31,7,0,0,NaN,0,4.92,...,0,0,0,0,0,0,0,0,0,1
1987893,68czcr4BxJyMQ9cJBm6C7Q,Jane,1,2016-06-14 07:20:52,0,0,0,NaN,0,5.00,...,0,0,0,0,0,0,0,0,0,1
1987894,1x3KMskYxOuJCjRz70xOqQ,Shomari,4,2017-02-04 15:31:58,1,1,0,NaN,0,2.00,...,0,0,0,0,0,0,0,0,0,1
1987895,ulfGl4tdbrH05xKzh5lnog,Susanne,2,2011-01-14 00:29:08,0,0,0,NaN,0,3.00,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
# merged_no_ids_copy = merged_no_ids.copy()
# merged_no_ids_copy = merged_no_ids_copy.drop(columns=["review_count_user", "yelping_since", "useful_user", "funny_user", "cool_user", "fans", "average_stars", "compliment_hot", "compliment_more", "compliment_profile", "compliment_cute", "compliment_list", "compliment_note", "compliment_plain", "compliment_cool", "compliment_funny", "compliment_writer", "compliment_photos", "number_of_friends", "useful_review", "funny_review", "cool_review", "stars_review"])
# merged_no_ids_copy

,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,Conveyor Belt Sushi_category,Australian_category,Sushi Bars_category,...,BestNightsThursday_attribute_False,BestNightsThursday_attribute_True,BestNightsThursday_attribute_no_info,DriveThru_attribute_False,DriveThru_attribute_True,DriveThru_attribute_no_info,BestNightsMonday_attribute_False,BestNightsMonday_attribute_True,BestNightsMonday_attribute_no_info,date
0,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.547730
1,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.510133
2,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513276
3,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.783217
4,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.933705
511134,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.932890
511135,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.951791
511136,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.943997


In [11]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(merged_no_ids)
merged_no_ids = pd.DataFrame(scaler.transform(merged_no_ids), columns=merged_no_ids.columns)
display(merged_no_ids)

,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,Conveyor Belt Sushi_category,Australian_category,Sushi Bars_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000063,0.000000,0.000000
1,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000852,0.000336,0.000000,0.000186,0.000277,0.000921,0.000921,0.001632,0.000356,0.012338
2,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000079,0.000040,0.000040,0.000314,0.000000,0.001134
3,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
4,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000336,0.001534,0.000407,0.000504,0.000700,0.000700,0.000816,0.000089,0.009337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000010,0.000000,0.000000,0.000000,0.000000,0.000000
511134,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000030,0.000080,0.000080,0.000063,0.000053,0.000333
511135,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
511136,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000085,0.000000,0.000000,0.000000,0.000000,0.000000,0.005069


In [12]:
merged_no_ids_copy = merged_no_ids.copy()
merged_no_ids_copy = merged_no_ids_copy.drop(columns=["review_count_user", "yelping_since", "useful_user", "funny_user", "cool_user", "fans", "average_stars", "compliment_hot", "compliment_more", "compliment_profile", "compliment_cute", "compliment_list", "compliment_note", "compliment_plain", "compliment_cool", "compliment_funny", "compliment_writer", "compliment_photos", "number_of_friends", "useful_review", "funny_review", "cool_review", "stars_review"])
merged_no_ids_copy

,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,Conveyor Belt Sushi_category,Australian_category,Sushi Bars_category,...,BestNightsThursday_attribute_False,BestNightsThursday_attribute_True,BestNightsThursday_attribute_no_info,DriveThru_attribute_False,DriveThru_attribute_True,DriveThru_attribute_no_info,BestNightsMonday_attribute_False,BestNightsMonday_attribute_True,BestNightsMonday_attribute_no_info,date
0,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.547730
1,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.510133
2,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513276
3,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.783217
4,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.933705
511134,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.932890
511135,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.951791
511136,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.943997


In [13]:
merged = merged_only_ids.join(merged_no_ids)
display(merged)

,review_id,user_id,business_id,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,BXQcBN0iAi1lAUxibGLFzA,6_SpY41LIHZuIaiDs5FMKA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000063,0.000000,0.000000
1,uduvUCvi9w3T2bSGivCfXg,tCXElwhzekJEH6QJe3xs7Q,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000852,0.000336,0.000000,0.000186,0.000277,0.000921,0.000921,0.001632,0.000356,0.012338
2,a0vwPOqDXXZuJkbBW2356g,WqfKtI-aGMmvbA9pPUxNQQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000079,0.000040,0.000040,0.000314,0.000000,0.001134
3,MKNp_CdR2k2202-c8GN5Dw,3-1va0IQfK-9tUMzfHWfTA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
4,D1GisLDPe84Rrk_R4X2brQ,EouCKoDfzaVG0klEgdDvCQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000336,0.001534,0.000407,0.000504,0.000700,0.000700,0.000816,0.000089,0.009337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,qBcwQEQPnLxjkw-xbUIF4Q,6nF5PT1c0dF6EpOgQdF2tw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000010,0.000000,0.000000,0.000000,0.000000,0.000000
511134,G8fbysnUAUmqq1XWTjMQ4Q,1M78_w4J9f5S8xmUVYyxdQ,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000030,0.000080,0.000080,0.000063,0.000053,0.000333
511135,JKiy0aeyGd3KmXN7uRPFLw,B7TD5yTemGv50y4wM2EVNA,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
511136,JITY01bGbdsiUBznLz9rdg,HI8QwhpeP_ZRY5JZy11VDw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000085,0.000000,0.000000,0.000000,0.000000,0.000000,0.005069


In [14]:
merged_copy = merged_only_ids.join(merged_no_ids_copy)
display(merged_copy)

,review_id,user_id,business_id,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,...,BestNightsThursday_attribute_False,BestNightsThursday_attribute_True,BestNightsThursday_attribute_no_info,DriveThru_attribute_False,DriveThru_attribute_True,DriveThru_attribute_no_info,BestNightsMonday_attribute_False,BestNightsMonday_attribute_True,BestNightsMonday_attribute_no_info,date
0,BXQcBN0iAi1lAUxibGLFzA,6_SpY41LIHZuIaiDs5FMKA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.547730
1,uduvUCvi9w3T2bSGivCfXg,tCXElwhzekJEH6QJe3xs7Q,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.510133
2,a0vwPOqDXXZuJkbBW2356g,WqfKtI-aGMmvbA9pPUxNQQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513276
3,MKNp_CdR2k2202-c8GN5Dw,3-1va0IQfK-9tUMzfHWfTA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.783217
4,D1GisLDPe84Rrk_R4X2brQ,EouCKoDfzaVG0klEgdDvCQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.513282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,qBcwQEQPnLxjkw-xbUIF4Q,6nF5PT1c0dF6EpOgQdF2tw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.933705
511134,G8fbysnUAUmqq1XWTjMQ4Q,1M78_w4J9f5S8xmUVYyxdQ,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.932890
511135,JKiy0aeyGd3KmXN7uRPFLw,B7TD5yTemGv50y4wM2EVNA,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.951791
511136,JITY01bGbdsiUBznLz9rdg,HI8QwhpeP_ZRY5JZy11VDw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.943997


In [15]:
merged.value_counts("user_id")

user_id
ET8n-r7glWYqZhuR6GcdNw    610
vFd8aBLg1kFcd0kCkoi-xw    447
_BcWyKQL16ndpBdggh2kNA    441
bJ5FtCtZX3ZZacz2_2PJjA    375
0DB3Irpf_ETVXu_Ou9vPow    357
                         ... 
ogGTGzUzKn1x5-ddTU4yfQ      1
YqiMF5KzIw8xP494HNuKdw      1
4rNNfQm1tB1xOVQ9ZduzNA      1
7Ka3Smzb7hlHi8gk0wByjw      1
B7TD5yTemGv50y4wM2EVNA      1
Name: count, Length: 178325, dtype: int64

In [16]:
indexes = merged.value_counts("user_id")[(merged.value_counts("user_id") == 5)].index
indexes

Index(['nDFRVVcNLLUt-F_s0yBIPA', 'STq_SEa2rTEGPgKSP388Sw',
       'H_CJLohiZzbnHWBI-P-u8Q', 'dbqRq6_KOKjcl2uT51DLKA',
       'Qth4n3N3XYzEgIIfXDVXww', '8mNhnIUBUTsi4BwkIyKUqw',
       'aRYk2snBNhxAk6hBgWD-Aw', 'QKlRvN4cJszeKI06usG6dA',
       '_rVlbC15nRQPSq-o3hMzdQ', 'EXFvkdyj-mhzm3PA2ESHLg',
       ...
       'GQgH--hxKJisGmfsKj0xDA', 'Zy8NOrdYcTaKJxbfdM0kHQ',
       'nh6ArLduHi_53eMPMW0fgw', 'OxaqY6xoDhFFxyICEX4QQQ',
       'EWLuooWEKiGZlxl3jccVyA', 'LU0KeX-SHEFhI77xqDueoA',
       'MKkCT6R15R2nrURE6VTOgA', 'kJModxtvXak455L9xo9xXw',
       'RhoZYeIa3tP0nTEitoNFzA', 'knE5x-gQ6ALjcXHpSkpkVw'],
      dtype='str', name='user_id', length=4269)

In [17]:
merged

,review_id,user_id,business_id,stars_business,review_count_review,Fuzhou_category,Themed Cafes_category,Burmese_category,Falafel_category,Cafes_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,BXQcBN0iAi1lAUxibGLFzA,6_SpY41LIHZuIaiDs5FMKA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000063,0.000000,0.000000
1,uduvUCvi9w3T2bSGivCfXg,tCXElwhzekJEH6QJe3xs7Q,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000852,0.000336,0.000000,0.000186,0.000277,0.000921,0.000921,0.001632,0.000356,0.012338
2,a0vwPOqDXXZuJkbBW2356g,WqfKtI-aGMmvbA9pPUxNQQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000079,0.000040,0.000040,0.000314,0.000000,0.001134
3,MKNp_CdR2k2202-c8GN5Dw,3-1va0IQfK-9tUMzfHWfTA,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
4,D1GisLDPe84Rrk_R4X2brQ,EouCKoDfzaVG0klEgdDvCQ,MTSW4McQd7CbVtyjqoe9mw,0.750,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000336,0.001534,0.000407,0.000504,0.000700,0.000700,0.000816,0.000089,0.009337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,qBcwQEQPnLxjkw-xbUIF4Q,6nF5PT1c0dF6EpOgQdF2tw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000010,0.000000,0.000000,0.000000,0.000000,0.000000
511134,G8fbysnUAUmqq1XWTjMQ4Q,1M78_w4J9f5S8xmUVYyxdQ,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000051,0.000030,0.000080,0.000080,0.000063,0.000053,0.000333
511135,JKiy0aeyGd3KmXN7uRPFLw,B7TD5yTemGv50y4wM2EVNA,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
511136,JITY01bGbdsiUBznLz9rdg,HI8QwhpeP_ZRY5JZy11VDw,WnT9NIzQgLlILjPT0kEcsQ,0.875,0.005248,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000085,0.000000,0.000000,0.000000,0.000000,0.000000,0.005069


In [18]:
user_matrix = np.array(
    [merged[merged["user_id"] == user_id].sort_values("date").drop(labels=["review_id", "user_id", "business_id"], axis=1).to_numpy() for user_id in indexes]
)

In [19]:
user_matrix_y = np.array(
    [merged_copy[merged_copy["user_id"] == user_id].sort_values("date").drop(labels=["review_id", "user_id", "business_id", "date"], axis=1).to_numpy() for user_id in indexes]
)

In [20]:
user_matrix.shape

(4269, 5, 385)

In [21]:
user_matrix_y.shape

(4269, 5, 361)

In [22]:
user_matrix.dtype

dtype('float64')

In [23]:
user_matrix_y.dtype

dtype('float64')

In [24]:
# user_matrix = user_matrix[~np.isnan(user_matrix)]
user_matrix

array([[[6.25000000e-01, 6.52554234e-02, 0.00000000e+00, ...,
         1.25517761e-04, 0.00000000e+00, 2.66773376e-04],
        [8.75000000e-01, 1.24212736e-02, 0.00000000e+00, ...,
         1.25517761e-04, 0.00000000e+00, 2.66773376e-04],
        [8.75000000e-01, 1.22463261e-03, 0.00000000e+00, ...,
         1.25517761e-04, 0.00000000e+00, 2.66773376e-04],
        [7.50000000e-01, 1.31210637e-02, 0.00000000e+00, ...,
         1.25517761e-04, 0.00000000e+00, 2.66773376e-04],
        [5.00000000e-01, 3.67389783e-02, 0.00000000e+00, ...,
         1.25517761e-04, 0.00000000e+00, 2.66773376e-04]],

       [[7.50000000e-01, 1.12841148e-01, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [5.00000000e-01, 5.37088873e-02, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [7.50000000e-01, 4.19874038e-02, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [7.50000000e-01, 6.77046886e-0

In [25]:
x = user_matrix[:, :4, :]
y = user_matrix_y[:, 4:, :]

In [26]:
display(x.shape)
display(y.shape)

(4269, 4, 385)

(4269, 1, 361)

In [23]:
display(0.1/(32*4*385))
display(1/(32*4*385))
display((0.1/(32*4*385)) + (1/(32*4*385)))

2.0292207792207792e-06

2.0292207792207792e-05

2.232142857142857e-05

In [27]:
model = Sequential([
    LSTM(512, activation="tanh", return_sequences=True, input_shape=(4, 385)),  # First LSTM layer
    LSTM(384, activation="tanh"),  # Second LSTM layer
    Dense(256, activation="relu"),
    # Dropout(0.1),
    Dense(361, activation="tanh"),
])
optimizer = keras.optimizers.Adam()
optimizer.learning_rate.assign(0.000001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
model.summary()

E0000 00:00:1776268746.931979   16412 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/kristofs/School/ISA/ISA_project/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 4, 512)         │     1,839,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 384)            │     1,377,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 361)            │        92,777 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,408,233 (13.00 MB)

 Trainable params: 3,408,233 (13.00 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
history = model.fit(x, y, epochs=300, batch_size=256, verbose=1)

Epoch 1/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 7s 148ms/step - loss: 0.2115 - mae: 0.2335
Epoch 2/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 128ms/step - loss: 0.2110 - mae: 0.2331
Epoch 3/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 119ms/step - loss: 0.2105 - mae: 0.2328
Epoch 4/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 116ms/step - loss: 0.2100 - mae: 0.2325
Epoch 5/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 118ms/step - loss: 0.2095 - mae: 0.2323
Epoch 6/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 116ms/step - loss: 0.2090 - mae: 0.2321
Epoch 7/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - loss: 0.2085 - mae: 0.2320
Epoch 8/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - loss: 0.2080 - mae: 0.2320
Epoch 9/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 116ms/step - loss: 0.2075 - mae: 0.2319
Epoch 10/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - loss: 0.2070 - mae: 0.2320
Epoch 11/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - loss: 0.2065 - mae: 0.2320
Epoch 12/300
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - loss: 0.2060 - mae: 0.2322
Epoch 13/300
